# Production Quality Monitoring

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/use-cases/production-quality-monitoring). Run the cells in order in a fresh Python 3.12 kernel.

## Install

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "agent-learning-kit==0.3.7", "fi-instrumentation-otel==1.5.1", "traceai-openai==0.1.16", "openai==2.54.0"], check=True);

## Set your credentials

Set the named environment variables or Colab secrets before running, or enter them at the hidden prompts. Values are not displayed. For a self-hosted deployment, set `FI_BASE_URL` to your API and `FI_TRACES_URL` to your collector before continuing.

In [ ]:
import os
from getpass import getpass


def require_secret(name):
    if os.environ.get(name):
        return
    value = None
    try:
        from google.colab import userdata
    except ImportError:
        pass
    else:
        try:
            value = userdata.get(name)
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            pass
    if not value:
        value = getpass(f"{name}: ")
    if not value:
        raise ValueError(f"{name} is required")
    os.environ[name] = value


for name in ('FI_API_KEY', 'FI_SECRET_KEY', 'OPENAI_API_KEY'):
    require_secret(name)

os.environ.setdefault("FI_TRACES_URL", "https://api.futureagi.com")

## Define the agent you want to monitor

In [ ]:
SYSTEM_PROMPT = """You are the Ridgeline Outfitters support assistant. For product or order questions, use the matching tool. Answer only from its results.
If you don't have the information, say so. Never guess or fabricate details."""

In [ ]:
import json
from openai import OpenAI

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_products",
            "description": "Search the catalog using one product keyword, such as lantern or mug",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_order_status",
            "description": "Look up order status by order ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "The order ID"},
                },
                "required": ["order_id"],
            },
        },
    },
]


def search_products(query: str) -> dict:
    catalog = [{"name": "Trail Lantern", "price": 79.99, "in_stock": True},
               {"name": "Camp Mug", "price": 15.00, "in_stock": True}]
    words = query.lower().split()
    matches = [p for p in catalog if any(
        word.rstrip("s") in p["name"].lower().split() for word in words
    )]
    if not matches:
        # The reply and its context evals both need the lookup outcome.
        return {"results": [], "message": "No catalog items matched this search."}
    return {"results": matches}


def get_order_status(order_id: str) -> dict:
    if order_id == "ORD-12345":
        return {"order_id": order_id, "status": "shipped", "carrier": "ParcelCo"}
    return {"order_id": order_id, "error": "No matching order"}


TOOL_MAP = {
    "search_products": search_products,
    "get_order_status": get_order_status,
}


def generate_reply(messages: list) -> tuple[str, str]:
    """Process a user message. Returns (answer, context_from_tools)."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": SYSTEM_PROMPT}] + messages,
        tools=TOOLS,
    )

    msg = response.choices[0].message
    context = ""

    if msg.tool_calls:
        tool_messages = [msg]
        tool_results = []
        for tool_call in msg.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            result = TOOL_MAP.get(fn_name, lambda **_: {"error": "Unknown tool"})(**fn_args)
            result_str = json.dumps(result)
            tool_results.append(result_str)
            tool_messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result_str,
            })

        context = "\n".join(tool_results)
        followup = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": SYSTEM_PROMPT}] + messages + tool_messages,
        )
        msg = followup.choices[0].message

    if not msg.content:
        raise RuntimeError("The model did not return a text answer")
    return msg.content, context

## Trace every call

In [ ]:
from fi_instrumentation import register, FITracer, using_user, using_session
from fi_instrumentation.fi_types import ProjectType
from traceai_openai import OpenAIInstrumentor

trace_provider = register(
    project_type=ProjectType.OBSERVE,
    project_name="ridgeline-quality-monitor",
    set_global_tracer_provider=True,
    verbose=False,
)
OpenAIInstrumentor().instrument(tracer_provider=trace_provider)
client = OpenAI()

tracer = FITracer(trace_provider.get_tracer("ridgeline-quality-monitor"))

In [ ]:
traced_reply = tracer.agent(name="support_assistant")(generate_reply)

def handle_message(user_id: str, session_id: str, messages: list) -> tuple[str, str]:
    with using_user(user_id), using_session(session_id):
        return traced_reply(messages)

In [ ]:
test_queries = [
    "Show me trail lanterns under $100",
    "Where is my order ORD-12345?",
    "What's your return policy?",
]

for i, query in enumerate(test_queries):
    answer, _ = handle_message(
        user_id=f"user-{100 + i}",
        session_id=f"session-{i}",
        messages=[{"role": "user", "content": query}],
    )
    print(f"Q: {query}")
    print(f"A: {answer[:120]}...\n")

trace_provider.force_flush()

## Score every response with inline evals

In [ ]:
from fi.evals import Evaluator
from opentelemetry import trace

evaluator = Evaluator()


def attach(metric, inputs):
    result = evaluator.evaluate(
        eval_templates=metric, inputs=inputs, model_name="turing_small",
    ).eval_results[0]
    if not result.eval_id or result.output is None:
        raise RuntimeError(f"No {metric} score: {result.reason}")
    span = trace.get_current_span()
    span.set_attribute(f"quality.{metric}.value", result.output)
    span.set_attribute(f"quality.{metric}.eval_id", str(result.eval_id))
    span.set_attribute(f"quality.{metric}.reason", result.reason or "")
    print(f"{metric}_check: {result.output}")
    return result.output


@tracer.agent(name="scored_assistant")
def score_reply(messages: list) -> str:
    answer, context = generate_reply(messages)
    question = messages[-1]["content"]
    attach("completeness", {"input": question, "output": answer})
    if context:
        attach("context_adherence", {"output": answer, "context": context})
        attach("context_relevance", {"input": question, "context": context})
    return answer


def handle_message_scored(user_id: str, session_id: str, messages: list) -> str:
    with using_user(user_id), using_session(session_id):
        return score_reply(messages)

In [ ]:
eval_queries = [
    "What trail lanterns do you have in stock?",
    "Has order ORD-12345 shipped?",
    "Is the Camp Mug in stock?",
    "Can I get a refund on a product I bought two months ago?",
    "What's the cheapest item in your catalog?",
]

for i, query in enumerate(eval_queries):
    answer = handle_message_scored(
        user_id=f"user-{200 + i}",
        session_id=f"eval-session-{i}",
        messages=[{"role": "user", "content": query}],
    )
    print(f"Q: {query}")
    print(f"A: {answer[:150]}...\n")

trace_provider.force_flush()

## Alert on quality drops

In the app, open **Alerts**, select your project and choose thresholds based on observed latency, failures or token spend. Running this notebook does not create an alert or send notifications.

## Cluster failures with Error Feed

In [ ]:
TOOL_MAP["search_products"] = tracer.tool(name="search_products")(search_products)
TOOL_MAP["get_order_status"] = tracer.tool(name="get_order_status")(get_order_status)
for query in ("Where is my order ORD-12345?", "Is the Camp Mug in stock?"):
    answer, context = handle_message("user-replay", "tool-replay", [{"role": "user", "content": query}])
    print("Tool evidence:", context)
    print("Reply:", answer)
trace_provider.force_flush()

## Gate the request and response on safety verdicts

In [ ]:
INPUT_FALLBACK = "Please ask about an order or product without including sensitive details."
OUTPUT_FALLBACK = "I can't share that response. Please contact support through your account."


def allowed(text, stage):
    if stage == "input":
        checks = [("data_privacy_compliance", "output"), ("prompt_injection", "input"),
                  ("toxicity", "output")]
    elif stage == "output":
        checks = [("data_privacy_compliance", "output"), ("toxicity", "output"),
                  ("bias_detection", "output")]
    else:
        raise ValueError("Expected input or output stage")

    with trace_provider.get_tracer(__name__).start_as_current_span(f"{stage}_guard"):
        for metric, field in checks:
            if attach(metric, {field: text}) != "Passed":
                return False
    return True


@tracer.agent(name="guarded_assistant")
def guarded_reply(messages: list) -> str:
    if not allowed(messages[-1]["content"], "input"):
        return INPUT_FALLBACK
    answer = score_reply(messages)
    return answer if allowed(answer, "output") else OUTPUT_FALLBACK

In [ ]:
for i, query in enumerate(("Is the Camp Mug in stock?",
                           "My full credit card number is 4111 1111 1111 1111. Can you find my order?")):
    with using_user(f"user-guard-{i}"), using_session(f"guard-{i}"):
        reply = guarded_reply([{"role": "user", "content": query}])
        print("Response:", reply)

draft = "The customer's full credit card number is 4111 1111 1111 1111."
print("Draft:", draft if allowed(draft, "output") else OUTPUT_FALLBACK)
trace_provider.force_flush()

Inspect the input and output guard spans in Observe. A blocked input should not make a model call, and a blocked draft should not reach the user.